# Kaggle 1/2 - Prepare the data (no GPU needed)

**Settings (right-hand panel):** Accelerator = **None** (this step does not need a GPU, so it does not use your GPU quota). Internet = **On**.

What this notebook does: downloads MSD Task01 (about 7 GB), converts it to nnU-Net format (ID 501), **holds out the test patients**, runs nnU-Net preprocessing, and packages what training needs into `/kaggle/working/prepared`.

Run the cells in order, one at a time. Then click **Save Version -> Save & Run All (Commit)** so the packaged output is stored and can be attached to notebook 2.

Working data goes to `/kaggle/temp` (large, but erased at the end of the session). Only the small packaged result goes to `/kaggle/working` (limit about 20 GB).

In [ ]:
# 0. Check disk space and internet. Stop here if either looks wrong.
!df -h /kaggle/temp /kaggle/working
!curl -sI https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar | head -n 1

In [ ]:
# 1. Get the code and install nnU-Net. Do NOT install torch yourself: Kaggle already has a matching one.
import os
if not os.path.exists('brats-nnunet'):
    !git clone https://github.com/Hidaayet/brats-nnunet
%cd brats-nnunet
!pip install -q nnunetv2 nibabel scipy pandas matplotlib tqdm

In [ ]:
# 2. Settings
CONFIGS = '3d_fullres'     # run this notebook again with '2d' later for the 2D baseline
os.environ['BASE'] = '/kaggle/temp/nnunet'

In [ ]:
# 3. Download -> convert -> hold out test patients -> preprocess (30-60 min, just wait)
!bash scripts/01_prepare_data.sh "$CONFIGS"

In [ ]:
# 4. Package only what training/testing needs into /kaggle/working/prepared
import shutil, pathlib
BASE = os.environ['BASE']; DS = 'Dataset501_BrainTumour'
P = pathlib.Path('/kaggle/working/prepared'); P.mkdir(exist_ok=True)
dst = P / 'nnUNet_preprocessed' / DS
if not dst.exists():
    shutil.copytree(f'{BASE}/nnUNet_preprocessed/{DS}', dst)
for sub in ['imagesTs_heldout', 'labelsTs_heldout']:
    if not (P / 'heldout' / sub).exists():
        shutil.copytree(f'{BASE}/nnUNet_raw/{DS}/{sub}', P / 'heldout' / sub)
shutil.copy('splits/split_v1.json', P / 'split_v1.json')
!du -sh /kaggle/working/prepared/*
!df -h /kaggle/working | tail -n 1

In [ ]:
# 5. IMPORTANT: freeze the split. Copy this file's content into splits/split_v1.json in your GitHub repo and commit it.
print(open('splits/split_v1.json').read()[:600], '...')

## Next
1. Click **Save Version -> Save & Run All (Commit)**. Wait until it finishes.
2. Open notebook 2, click **Add Input -> Your Work (or Notebook Output Files) -> this notebook**.
3. If the size check in cell 4 showed more than about 15 GB, run only one configuration per notebook (that is why `CONFIGS` is a variable).